# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ameen740/Internship_Flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My method is Logistic Regression because it is a simple and interpretable model for learning patterns from search-performance signals. It fits my lane because I want to identify content items that may need search-performance attention and compare the learned model with my Week-4 rule-based baseline.

The model will use signals that are available at the decision moment, such as search impressions, clicks, CTR, and average search position. I will not use client or content identifiers as predictive features, and I will exclude future-window or label-derived information to avoid data leakage.

I chose Logistic Regression because the goal is not to use the most complex model, but to test whether a simple learned model can provide better decision support than my existing baseline.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 — Question 1: Method choice

method = "Logistic Regression"

features = [
    "gsc_impressions",
    "gsc_clicks",
    "CTR",
    "gsc_avg_position"
]

excluded = [
    "client_hash_id",
    "content_hash_id",
    "future-window information",
    "label-derived information"
]

print("Selected method:", method)
print("Model features:")
for feature in features:
    print("-", feature)

print("\nExcluded:")
for item in excluded:
    print("-", item)


Selected method: Logistic Regression
Model features:
- gsc_impressions
- gsc_clicks
- CTR
- gsc_avg_position

Excluded:
- client_hash_id
- content_hash_id
- future-window information
- label-derived information


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a grouped train-test split based on client_hash_id. The client identifier will be used only for creating the groups and will not be used as a model feature.

I will keep all rows from the same client in either the training set or the test set, rather than putting the same client's rows in both sets. This is more honest because records from the same client can have similar search-performance patterns.

I will use 80% of the client groups for training and 20% for testing. The Week-5 model and the Week-4 baseline will use the same test data and evaluation metric so their results can be compared fairly.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 - Load March 2026 Search Intelligence data

import duckdb
import pandas as pd
from google.colab import userdata

# Get the Hugging Face token from the Colab Secret
HF_TOKEN = userdata.get("HF_Token")

# Connect to DuckDB
con = duckdb.connect()

# Give DuckDB access to Hugging Face
con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')"
)

# Hugging Face warehouse location
warehouse = "hf://datasets/FlyRank/internship-warehouse"

# Load only March 2026 data
query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    gsc_data_available
FROM read_parquet(
    '{warehouse}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
"""

df = con.sql(query).df()

print("Data loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data loaded successfully!
Rows: 9841378
Columns: 7

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_data_available']

First 5 rows:


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,gsc_data_available
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,67,True
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0,True
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,616,True
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,28,True
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,25,True


In [24]:

# TO Check the DF is exists or not
print("df exists:", "df" in globals())
print("Rows:", len(df))
print("Unique clients:", df["client_hash_id"].nunique())

df exists: True
Rows: 9841378
Unique clients: 55


In [25]:
# This is the Actual code of Question NO 2

# ML-08 — Question 2: Grouped train-test split

from sklearn.model_selection import GroupShuffleSplit

# Use the loaded warehouse data
model_df = df.copy()

print("Total rows:", len(model_df))
print("Unique clients:", model_df["client_hash_id"].nunique())

# 80% of clients for training, 20% for testing
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_df,
        groups=model_df["client_hash_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

# Check client overlap
train_clients = set(train_df["client_hash_id"])
test_clients = set(test_df["client_hash_id"])

overlap = train_clients.intersection(test_clients)

print("\nTrain rows:", len(train_df))
print("Test rows:", len(test_df))

print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))

print("Client overlap:", len(overlap))

if len(overlap) == 0:
    print("\nPASS: No client appears in both train and test.")
else:
    print("\nWARNING: Client overlap detected.")

Total rows: 9841378
Unique clients: 55

Train rows: 8935676
Test rows: 905702
Train clients: 44
Test clients: 11
Client overlap: 0

PASS: No client appears in both train and test.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I will compare Logistic Regression with my Week-4 rule-based baseline using the same March 2026 data and the same grouped client split. The Week-4 baseline uses impressions and CTR to identify CTR review candidates. For this modeling exercise, the baseline review decision will be used as the target so that the learned model can be trained and evaluated consistently. I will use the same observable search-performance signals as model features while excluding client and content identifiers. The comparison is intended to show how closely the learned model reproduces the baseline decision and whether it provides useful decision support; it is not evidence of causal improvement.


In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 — Question 3: Train Logistic Regression and compare with baseline

import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, accuracy_score


# 1. Prepare the modeling data

model_df = df.copy()

# Calculate CTR
model_df["ctr_pct"] = (
    100.0 * model_df["gsc_clicks"] / model_df["gsc_impressions"]
)

# Week-4 baseline decision
model_df["baseline_target"] = (
    (model_df["gsc_impressions"] >= 100) &
    (model_df["ctr_pct"] < 2.0)
).astype(int)

print("Total rows:", len(model_df))
print("Positive baseline cases:", model_df["baseline_target"].sum())
print("Negative baseline cases:", (model_df["baseline_target"] == 0).sum())

# 2. Use the same grouped train/test split from Q2

# Recreate the split if train_df/test_df already exist
# from Question 2, otherwise create it here.

from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_df,
        groups=model_df["client_hash_id"]
    )
)

train_model = model_df.iloc[train_idx].copy()
test_model = model_df.iloc[test_idx].copy()

# 3. Select model features

feature_names = [
    "gsc_impressions",
    "gsc_clicks",
    "ctr_pct"
]

X_train = train_model[feature_names].replace(
    [np.inf, -np.inf], np.nan
).fillna(0)

X_test = test_model[feature_names].replace(
    [np.inf, -np.inf], np.nan
).fillna(0)

y_train = train_model["baseline_target"]
y_test = test_model["baseline_target"]

# 4. Train Logistic Regression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# 5. Evaluate Logistic Regression

model_precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

model_accuracy = accuracy_score(
    y_test,
    y_pred
)

# 6. Evaluate the Week-4 baseline on the same test set

baseline_pred = test_model["baseline_target"]

baseline_precision = precision_score(
    y_test,
    baseline_pred,
    zero_division=0
)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_pred
)

# 7. Comparison table

comparison = pd.DataFrame({
    "Method": [
        "Week-4 Rule-Based Baseline",
        "Week-5 Logistic Regression"
    ],
    "Precision": [
        baseline_precision,
        model_precision
    ],
    "Accuracy": [
        baseline_accuracy,
        model_accuracy
    ]
})

print("\nMODEL VS BASELINE")
display(comparison)

# 8. Show model coefficients

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": model.coef_[0]
}).sort_values(
    "Coefficient",
    key=abs,
    ascending=False
)

print("\nLOGISTIC REGRESSION FEATURE INFLUENCE")
display(feature_importance)

# 9. Basic checks for Q4

print("\nVariables created for Question 4:")
print("y_test:", len(y_test))
print("y_pred:", len(y_pred))
print("model: trained")
print("feature_names:", feature_names)



Total rows: 9841378
Positive baseline cases: 628945
Negative baseline cases: 9212433

MODEL VS BASELINE


,Method,Precision,Accuracy
0,Week-4 Rule-Based Baseline,1.0000,1.000000
1,Week-5 Logistic Regression,0.8952,0.992887



LOGISTIC REGRESSION FEATURE INFLUENCE


,Feature,Coefficient
1,gsc_clicks,-1.588469
0,gsc_impressions,0.182258
2,ctr_pct,0.153093



Variables created for Question 4:
y_test: 905702
y_pred: 905702
model: trained
feature_names: ['gsc_impressions', 'gsc_clicks', 'ctr_pct']


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Logistic Regression model makes some incorrect predictions when compared with the Week-4 baseline decision. I will examine these errors to identify the types of search-performance cases where the model disagrees with the baseline. The model coefficients show that clicks have the strongest influence among the three features, followed by impressions and CTR based on absolute coefficient size. This means the model relies mainly on the observed search-performance signals, especially clicks. Because the model does not perfectly reproduce the baseline, its predictions should be treated as decision support and reviewed before action.


In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-08 — Question 4: Errors and interpretation

# 1. Identify incorrect predictions


errors = (y_test != y_pred)

print("ERROR ANALYSIS")
print("Total test examples:", len(y_test))
print("Incorrect predictions:", errors.sum())
print("Error rate:", round(errors.mean() * 100, 2), "%")


# 2. Show examples of incorrect predictions

error_rows = test_model.loc[errors].copy()

print("\nNumber of incorrect rows:", len(error_rows))

error_columns = [
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "ctr_pct",
    "baseline_target"
]

available_columns = [
    col for col in error_columns
    if col in error_rows.columns
]

display(
    error_rows[available_columns].head(20)
)


# 3. Compare actual vs predicted decisions

print("\nACTUAL VS PREDICTED")

error_summary = pd.crosstab(
    y_test,
    y_pred,
    rownames=["Actual"],
    colnames=["Predicted"]
)

display(error_summary)

# 4. Feature influence

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": model.coef_[0]
})

feature_importance["Absolute_Influence"] = (
    feature_importance["Coefficient"].abs()
)

feature_importance = feature_importance.sort_values(
    "Absolute_Influence",
    ascending=False
)

print("\nFEATURE INFLUENCE")
display(feature_importance)


# 5. Final interpretation

print("\nINTERPRETATION")
print("The largest absolute coefficient shows the strongest model influence.")
print("The model does not perfectly reproduce the Week-4 baseline.")
print("Errors should be reviewed as decision-support cases.")


ERROR ANALYSIS
Total test examples: 905702
Incorrect predictions: 6442
Error rate: 0.71 %

Number of incorrect rows: 6442


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr_pct,baseline_target
18984,client_c182d11e4862a37d,content_f24835a19fff49ee,94,0,0.0,0
18998,client_c182d11e4862a37d,content_e2bf3fd438472dbf,96,0,0.0,0
19129,client_c182d11e4862a37d,content_12510268ee8809ee,89,0,0.0,0
19141,client_c182d11e4862a37d,content_35031cb016d829eb,97,0,0.0,0
19170,client_c182d11e4862a37d,content_b8342bb16248c921,92,0,0.0,0
19175,client_c182d11e4862a37d,content_3afdcaa0cf9461be,89,0,0.0,0
19224,client_c182d11e4862a37d,content_08a38f9aac063491,91,0,0.0,0
19226,client_c182d11e4862a37d,content_41267db09926fb7b,88,0,0.0,0
19314,client_c182d11e4862a37d,content_98648444f3cc6e80,91,0,0.0,0
19398,client_c182d11e4862a37d,content_7e58eb5e3aa4a9bb,90,0,0.0,0



ACTUAL VS PREDICTED


Predicted,0,1
Actual,,
0,844651,6393
1,49,54609



FEATURE INFLUENCE


,Feature,Coefficient,Absolute_Influence
1,gsc_clicks,-1.588469,1.588469
0,gsc_impressions,0.182258,0.182258
2,ctr_pct,0.153093,0.153093



INTERPRETATION
The largest absolute coefficient shows the strongest model influence.
The model does not perfectly reproduce the Week-4 baseline.
Errors should be reviewed as decision-support cases.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.